# Topic Modelling

## Overview

Topic modelling is an unsupervised method for discovering latent themes in a document collection. It is useful for exploring large corpora of reports, literature, or field notes without manual labelling.

**Main approaches:**

| Method | Basis | Strengths |
|---|---|---|
| LDA (Latent Dirichlet Allocation) | Probabilistic generative model | Interpretable topics, soft membership |
| NMF (Non-negative Matrix Factorisation) | Matrix decomposition | Fast, often sharper topics than LDA |
| BERTopic | Sentence embeddings + clustering | State-of-the-art coherence |

**Key outputs:**
- Topic-term matrix: which words define each topic
- Document-topic matrix: how much each document belongs to each topic
- Coherence score: how semantically consistent the top words within a topic are

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(42)
# Simulate ecological report corpus with 3 underlying topics
topic_vocab = {
    'water_quality': [
        "nitrate phosphorus turbidity dissolved oxygen pH conductivity",
        "nutrient loading runoff agricultural contamination threshold exceeded",
        "water quality monitoring chemical parameters concentration elevated",
        "pollution source upstream discharge effluent water treatment",
    ],
    'biodiversity': [
        "species richness macroinvertebrate EPT taxa diversity index",
        "fish community salmonid invertebrate biological assessment",
        "abundance distribution ecological status habitat quality",
        "biodiversity recovery reference condition benchmark ecological",
    ],
    'restoration': [
        "riparian restoration buffer strip vegetation planting",
        "wetland constructed treatment removal phosphorus nitrogen",
        "habitat improvement channel realignment flood plain reconnection",
        "restoration project outcome monitoring success recovery progress",
    ],
}
# Generate corpus by mixing topics
corpus = []
topic_labels = []
for i in range(120):
    topic = rng.choice(list(topic_vocab.keys()))
    phrases = topic_vocab[topic]
    doc = ' '.join(rng.choice(phrases, size=rng.integers(2,4), replace=True))
    corpus.append(doc)
    topic_labels.append(topic)
print(f"Corpus: {len(corpus)} documents, {len(set(topic_labels))} true topics")
print(f"\nSample document: {corpus[0]}")

---
## LDA Topic Modelling

In [ ]:
# Count vectoriser for LDA (LDA expects raw counts, not TF-IDF)
cv = CountVectorizer(max_features=200, min_df=2, stop_words='english')
X_counts = cv.fit_transform(corpus)
n_topics = 3
lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=20,
    learning_method='online',
    random_state=42,
    n_jobs=-1,
)
lda.fit(X_counts)
feature_names = cv.get_feature_names_out()
def print_top_words(model, feature_names, n_top=8):
    for topic_idx, topic in enumerate(model.components_):
        top_idx  = topic.argsort()[-n_top:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        print(f"  Topic {topic_idx}: {' | '.join(top_words)}")
print("LDA Topics (top 8 words each):")
print_top_words(lda, feature_names)
print(f"\nPerplexity: {lda.perplexity(X_counts):.1f}  (lower = better fit)")

---
## NMF Topic Modelling

In [ ]:
# NMF uses TF-IDF (not counts)
tfidf_vec = TfidfVectorizer(max_features=200, min_df=2, stop_words='english')
X_tfidf = tfidf_vec.fit_transform(corpus)
nmf = NMF(n_components=n_topics, random_state=42, max_iter=400,
          init='nndsvda', l1_ratio=0.5)
nmf.fit(X_tfidf)
tfidf_features = tfidf_vec.get_feature_names_out()
print("NMF Topics (top 8 words each):")
print_top_words(nmf, tfidf_features)
# Compare reconstruction error
print(f"\nNMF reconstruction error: {nmf.reconstruction_err_:.4f}")
# Document-topic assignments
doc_topics_lda = lda.transform(X_counts)
doc_topics_nmf = nmf.transform(X_tfidf)
print(f"\nDocument-topic matrix shape: {doc_topics_lda.shape}")
print(f"Each row sums to 1 (LDA): {doc_topics_lda[0].sum():.3f}")
print(f"\nSample document topic distribution (LDA):")
for i, (doc, true_t) in enumerate(zip(corpus[:3], topic_labels[:3])):
    dist = doc_topics_lda[i]
    print(f"  Doc {i} (true: {true_t:12s}): {dist.round(3)}")

---
## Choosing the Number of Topics

In [ ]:
# Perplexity vs n_topics (elbow method)
perplexities = []
topic_range = range(2, 8)
for k in topic_range:
    lda_k = LatentDirichletAllocation(n_components=k, max_iter=15,
                                       learning_method='online', random_state=42)
    lda_k.fit(X_counts)
    perplexities.append(lda_k.perplexity(X_counts))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(list(topic_range), perplexities, 'o-', color='steelblue', lw=2, ms=8)
ax.axvline(3, color='#e74c3c', lw=1.5, linestyle='--', label='True k=3')
ax.set_xlabel('Number of topics (k)')
ax.set_ylabel('Perplexity (lower = better fit)')
ax.set_title('LDA Perplexity vs Number of Topics')
ax.legend(); plt.tight_layout(); plt.show()
print("Elbow or levelling-off in perplexity suggests optimal k")
print("Also consider topic coherence (C_v) using gensim's CoherenceModel")
print("Human interpretation of top words is essential -- numbers alone are insufficient")

In [ ]:
# Topic heatmap: document-topic distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
topic_names = ['Topic 0','Topic 1','Topic 2']
true_order = sorted(range(len(corpus)), key=lambda i: topic_labels[i])
for ax, doc_topics, title in [(axes[0], doc_topics_lda, 'LDA'),
                               (axes[1], doc_topics_nmf/doc_topics_nmf.sum(axis=1,keepdims=True), 'NMF')]:
    im = ax.imshow(doc_topics[true_order[:40]].T, aspect='auto',
                   cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_yticks(range(n_topics))
    ax.set_yticklabels(topic_names)
    ax.set_xlabel('Document index (sorted by true topic)')
    ax.set_title(f'{title} Document-Topic Matrix
(rows=topics, cols=docs)')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
# Dominant topic per document
dominant_lda = doc_topics_lda.argmax(axis=1)
dominant_nmf = doc_topics_nmf.argmax(axis=1)
print("True topic distribution (first 12 docs):")
for i in range(12):
    print(f"  Doc {i:2d}: true={topic_labels[i]:12s}, LDA_topic={dominant_lda[i]}, NMF_topic={dominant_nmf[i]}")

---

## Common Pitfalls

**1. Choosing the number of topics by perplexity alone**  
Perplexity measures how well the model predicts held-out documents — it generally decreases as k increases without a clear elbow. Topic coherence (C_v score, available in gensim) better reflects human interpretability. The final check must always be qualitative: can you give each topic a meaningful label?

**2. Using TF-IDF as input to LDA**  
LDA is a generative model of word counts — it assumes integer-valued term frequencies. TF-IDF produces real-valued weights that violate this assumption. Use `CountVectorizer` for LDA and `TfidfVectorizer` for NMF.

**3. Not removing domain stopwords before topic modelling**  
General English stopwords ("the", "is") are usually filtered, but domain-specific high-frequency terms (e.g. "site", "monitoring", "water" in ecological reports) appear in every topic and drown out discriminative terms. Always add domain stopwords to the stopword list.

**4. Interpreting topics that share many top words as distinct**  
If two topics share 5 of their top 8 words, they are not discovering meaningfully different themes — the model may be over-splitting one coherent topic. Reduce k or increase the regularisation prior.

**5. Treating topic assignments as ground-truth labels**  
Topic model outputs are soft probability distributions, not hard labels. The dominant topic is an approximation. Using topic assignments as features in a downstream supervised model without acknowledging their uncertainty can inflate apparent model performance.
---
*python_methods_library - Samantha McGarrigle*